# Calibration times over the study — SPX Legendre

Inspect your own saved runs, plot their times in run order, and compare early
and late calibrations. **Run All** reads the checkpoints and displays plots and
tables; it performs no calibrations and exports no files.

Run numbers identify accepted calibrations, not elapsed calendar time. Rejected
attempts are absent. Any timing-allowance changes recorded in the manifest are
marked on the plots and should be considered when comparing accepted runs.

SPX inputs and numerical results are not distributed. Run the calibration
notebooks with your own data before using this analysis.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "wiener_chaos/core/model.py").is_file())
sys.path.insert(0, str(ROOT))
from wiener_chaos.experiments.reporting import resolve_study_directory
STUDY_DIR = resolve_study_directory(
    ROOT / "Real data" / "Legendre" / "Statistics calibration" / "study")
ROLLING_RUNS = 10       # Trailing median in the plots.
COMPARE_RUNS = 25       # First and last 25 runs; automatically reduced for partial studies.

plt.rcParams.update({"figure.figsize": (12, 7), "axes.grid": True, "grid.alpha": .25})


## Load the individual checkpoints

- **Initialization:** Heston fitting and projection.
- **Calibration:** the saved chaos-calibration time, including setup and final pricing.
- **Total:** initialization plus calibration.
- **Measured training speed:** completed iterations divided by the sum of the
  saved 25-step window times. This uses the corrected speed timer, excluding
  setup and final evaluation. It is not total calibration time divided by iterations.

The number of iterations can vary with convergence, so longer calibration time
alone does not establish slower computation.


In [ ]:
if not (STUDY_DIR / "manifest.json").is_file():
    raise FileNotFoundError(
        "SPX study results are private and are not distributed. "
        "Run the Legendre calibration study with your own input data first."
    )

manifest = json.loads((STUDY_DIR / "manifest.json").read_text())
paths = sorted((STUDY_DIR / "runs").glob("run_*/result.json"))
if not paths:
    raise ValueError(f"No completed calibrations in {STUDY_DIR}")

initial_allowance = manifest["experiment"]["timing_guard"]["tolerance"]
rows = []
for path in paths:
    record = json.loads(path.read_text())
    if record["study_signature"] != manifest["study_signature"]:
        raise ValueError(f"Checkpoint belongs to a different study: {path}")
    profile = record.get("timing_profile", {})
    if profile.get("version") != 2 or not profile.get("windows"):
        raise ValueError(f"Corrected version-2 timing windows are required: {path}")
    windows = profile["windows"]
    interval = profile["check_every"]
    window_seconds = np.asarray([window["seconds"] for window in windows], dtype=float)
    if not np.isfinite(window_seconds).all() or (window_seconds <= 0).any():
        raise ValueError(f"Invalid timing window in {path}")
    allowance = record.get("timing_tolerance", initial_allowance)
    rows.append({
        "Run": record["run"],
        "Initialization (s)": record["elapsed_seconds_init"],
        "Calibration (s)": record["elapsed_seconds_chaos_calibration"],
        "Total (s)": record["elapsed_seconds_total"],
        "Iterations": record["calibration_iterations"],
        "Training speed (it/s)": interval * len(windows) / window_seconds.sum(),
        "Timing allowance (%)": 100 * allowance,
    })

runs = pd.DataFrame(rows).sort_values("Run").set_index("Run")
if not runs.index.is_unique:
    raise ValueError("Duplicate run numbers in the saved checkpoints.")
if not np.isfinite(runs.to_numpy(dtype=float)).all() or (runs <= 0).any().any():
    raise ValueError("The saved timings or iteration counts contain invalid values.")
if not np.allclose(runs["Total (s)"], runs["Initialization (s)"] + runs["Calibration (s)"]):
    raise ValueError("Saved total time is inconsistent with initialization + calibration.")
print(f"Loaded {len(runs)} completed calibrations: runs {runs.index.min()}–{runs.index.max()}.")
print(f"Study: {STUDY_DIR}")


## How do the times change with run number?

Dots show each accepted calibration; the thicker line is a trailing 10-run median.
The fourth panel compares measured iteration speed independently of how many
iterations were needed to converge.


In [ ]:
metrics = ["Initialization (s)", "Calibration (s)", "Total (s)", "Training speed (it/s)"]
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), sharex=True)
for ax, metric in zip(axes.flat, metrics):
    values = runs[metric]
    ax.plot(runs.index, values, ".-", linewidth=.8, markersize=4, alpha=.6, label="Individual run")
    ax.plot(runs.index, values.rolling(ROLLING_RUNS, min_periods=1).median(),
            linewidth=2, label=f"Trailing {ROLLING_RUNS}-run median")
    for change in manifest.get("timing_tolerance_changes", []):
        boundary = change["completed_runs"] + .5
        if runs.index.min() < boundary < runs.index.max():
            ax.axvline(boundary, color="tab:red", linestyle="--", linewidth=1.2,
                       label=f"Restart: allowance {100*change['from']:g}% → {100*change['to']:g}%")
    ax.set(ylabel=metric, xlabel="Calibration run")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Saved calibration times and measured training speed")
fig.tight_layout()
plt.show()


## Beginning versus end

Compare equally sized groups from the beginning and end. Positive percentage
changes mean longer times, more iterations, or faster training, depending on
the row. These are descriptive comparisons of the accepted runs.


In [ ]:
group_size = min(COMPARE_RUNS, len(runs) // 2)
if group_size < 1:
    raise ValueError("At least two completed runs are needed for an early/late comparison.")
early = runs.iloc[:group_size]
late = runs.iloc[-group_size:]
comparison_metrics = ["Initialization (s)", "Calibration (s)", "Total (s)",
                      "Iterations", "Training speed (it/s)"]
early_mean = early[comparison_metrics].mean()
late_mean = late[comparison_metrics].mean()
early_median = early[comparison_metrics].median()
late_median = late[comparison_metrics].median()
comparison = pd.DataFrame({
    f"First {group_size}: mean": early_mean,
    f"Last {group_size}: mean": late_mean,
    "Mean change (%)": 100 * (late_mean / early_mean - 1),
    f"First {group_size}: median": early_median,
    f"Last {group_size}: median": late_median,
    "Median change (%)": 100 * (late_median / early_median - 1),
})
display(comparison.round(3))
print(f"Compared runs {early.index.min()}–{early.index.max()} "
      f"with {late.index.min()}–{late.index.max()}.")
for metric in ("Calibration (s)", "Iterations", "Training speed (it/s)"):
    print(f"{metric}: {comparison.loc[metric, 'Mean change (%)']:+.2f}% in the mean.")


## Separate convergence length from computation speed

Each point is one calibration, colored by run number. The horizontal axis shows
how many iterations it needed. A shift toward more iterations can increase total
calibration time even when the measured iterations/s stays similar.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
points = ax.scatter(runs["Iterations"], runs["Calibration (s)"], c=runs.index,
                    cmap="viridis", s=32, alpha=.85)
fig.colorbar(points, ax=ax, label="Calibration run")
ax.set(xlabel="Optimizer iterations", ylabel="Calibration time (s)",
       title="Calibration time versus number of iterations")
fig.tight_layout()
plt.show()


## Compare timing allowances

Group your accepted runs by the timing allowance saved in their checkpoints.
This complements the early/late comparison without attributing differences to
a particular cause.


In [ ]:
periods = []
for allowance, group in runs.groupby("Timing allowance (%)", sort=False):
    periods.append({
        "Allowance (%)": allowance,
        "Runs": f"{group.index.min()}–{group.index.max()}",
        "Completed": len(group),
        "Mean initialization (s)": group["Initialization (s)"].mean(),
        "Mean calibration (s)": group["Calibration (s)"].mean(),
        "Mean iterations": group["Iterations"].mean(),
        "Mean training speed (it/s)": group["Training speed (it/s)"].mean(),
    })
display(pd.DataFrame(periods).set_index("Allowance (%)").round(3))


## Individual calibration times

Full table in calibration order. All values come from the saved accepted runs.


In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(runs.round(3))
